In [1]:
!nvidia-smi


Thu Nov  5 05:09:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers torch einops acccelerate pydrive2

ERROR: Could not find a version that satisfies the requirement acccelerate (from versions: none)
ERROR: No matching distribution found for acccelerate


In [3]:
%pip install pydrive2

In [4]:
%pip install transformers torch pillow einops accelerate pydrive2

In [ ]:
import json

# 1. Paste the ENTIRE contents of client_secrets.json here:
client_secrets_text = """

"""

# 2. Paste the ENTIRE contents of saved_creds.json here:
saved_creds_text = """

"""

# 3. Write them to the Colab Linux environment
with open("/content/client_secrets.json", "w") as f:
    f.write(client_secrets_text)

with open("/content/saved_creds.json", "w") as f:
    f.write(saved_creds_text)

print("[OK] Credentials created in Colab memory!")

[OK] Credentials created in Colab memory!


In [7]:
import json
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# 1. Paste your new saved_creds.json text here
saved_creds_text = """
{"access_token": "<YOUR_ACCESS_TOKEN>", "client_id": "<YOUR_CLIENT_ID>.apps.googleusercontent.com", "client_secret": "<YOUR_CLIENT_SECRET>", "refresh_token": null, "token_expiry": "2026-11-05T06:11:00Z", "token_uri": "https://oauth2.googleapis.com/token", "user_agent": null, "revoke_uri": "https://oauth2.googleapis.com/revoke", "id_token": null, "id_token_jwt": null, "token_response": {"access_token": "<YOUR_ACCESS_TOKEN>", "expires_in": 3599, "scope": "https://www.googleapis.com/auth/drive", "token_type": "Bearer"}, "scopes": ["https://www.googleapis.com/auth/drive"], "token_info_uri": "https://oauth2.googleapis.com/tokeninfo", "invalid": false, "_class": "OAuth2Credentials", "_module": "oauth2client.client"}
"""

# 2. Write it to the Colab memory
with open("/content/saved_creds.json", "w") as f:
    f.write(saved_creds_text)

# 3. Connect to Drive immediately
gauth = GoogleAuth()
gauth.LoadClientConfigFile("/content/client_secrets.json")
gauth.LoadCredentialsFile("/content/saved_creds.json")

if gauth.access_token_expired:
    gauth.Refresh()
else:
    gauth.Authorize()

drive = GoogleDrive(gauth)
print("[OK] Successfully connected to Google Drive!")

[OK] Successfully connected to Google Drive!


In [8]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
# Colab uses Linux paths, so we use forward slashes here
gauth.LoadClientConfigFile("/content/client_secrets.json") 
gauth.LoadCredentialsFile("/content/saved_creds.json")

if gauth.access_token_expired:
    gauth.Refresh()
else:
    gauth.Authorize()

drive = GoogleDrive(gauth)
print("[OK] Successfully connected to Google Drive!")

[OK] Successfully connected to Google Drive!


In [9]:
%pip install -q "transformers>=4.51.1" "torch>=2.7.0" "accelerate>=1.10.0" "Pillow>=11.0.0" einops

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("[OK] Loading Moondream VLM on the Cloud GPU...")
model_id = "vikhyatk/moondream2"

# Use 'dtype' (the new parameter name) and let it load the latest compatible weights
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    device_map={"": "cuda"}, 
    dtype=torch.float16 
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("[OK] Model loaded successfully on GPU!")

[OK] Loading Moondream VLM on the Cloud GPU...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

hf_moondream.py:   0%|          | 0.00/6.56k [00:00<?, ?B/s]

moondream.py:   0%|          | 0.00/35.5k [00:00<?, ?B/s]

text.py:   0%|          | 0.00/6.85k [00:00<?, ?B/s]

... (23 outputs left out)

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

[OK] Model loaded successfully on GPU!


In [11]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LoadClientConfigFile("/content/client_secrets.json")
gauth.LoadCredentialsFile("/content/saved_creds.json")

if gauth.access_token_expired:
    gauth.Refresh()
else:
    gauth.Authorize()

drive=GoogleDrive(gauth)
print('[ok] Sucessfully connected to googledrive!')

[ok] Sucessfully connected to googledrive!


In [ ]:
import time
from PIL import Image

DRIVE_FOLDER_ID ='1mzVbuOuXP9cGJksKTIFb8fnQZgf9amE5'
processed_file = set()

print('[ok] colab GPU is listening to google drive...')
print('waiting for new drone images to process...')

while True:
    try:
        file_list = drive.ListFile({'q': f"'{DRIVE_FOLDER_ID}' in parents and trashed=false"}).GetList()

        for file in file_list:
            if file['title'].endswith('.jpg') and file['title'] not in processed_file:
                print(f'\n[NEW EVENT ] processing:{file['title']}')

                file.GetContentFile('temp_drone_img.jpg')
                image = Image.open('temp_drone_img.jpg').convert('RGB')

                prompt = "Describe exactly what uoy see in this image"
                start_time = time.time()

                enc_iamge =model.encode_image(image)
                answer = model.answer_question(enc_image, prompt,tokenizer)

                print(f'----VLM RESPONSE----')
                print(answer)
                print(f'---------------')
                print(f'time taken:{time.time() - start_time:.2f} seconds')


                processed_file.add(file['title'])

    except Exception as e:
        print(f'[error] Drive check failed :{e}')

    time.sleep(5)






[ok] colab GPU is listening to google drive...
waiting for new drone images to process...

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed :name 'enc_image' is not defined

[NEW EVENT ] processing:track8_1792821718_cell phone.jpg
[error] Drive check failed